# Conferência dos 12 pares — registros documentais e relatos de escuta

Caderno de leitura dos registros preservados, sem consultas de rede ou audição pelo assistente. As comparações humanas, seus relatos e suas decisões estão em `recording-verification-evidence.json` e na planilha CSV. A origem de cada relato é explícita; campos e detalhes não informados permanecem ausentes.

Execute a partir de `notebooks/` ou da raiz do repositório. As evidências Spotify incluem horário e hash da consulta. Títulos YouTube são observações anteriores; as tentativas novas falharam. O cadastro principal e o limiar do piloto não foram alterados.

In [1]:
from collections import Counter
import csv
import hashlib
import json
from pathlib import Path

root = Path.cwd()
if not (root / 'data/pilot/metadata-review.json').exists():
    root = root.parent

baseline = json.loads((root / 'data/pilot/metadata-review.json').read_text())
evidence = json.loads((root / 'data/pilot/recording-verification-evidence.json').read_text())
with (root / 'data/pilot/recording-verification.csv').open(newline='') as file:
    worksheet = list(csv.DictReader(file))

expected = {record['pilot_id'] for record in baseline['reviews']}
assert len(expected) == 12
assert len(evidence['records']) == len(worksheet) == 12
assert {record['pilot_id'] for record in evidence['records']} == expected
assert {record['pilot_id'] for record in worksheet} == expected
for relative_path, expected_hash in evidence['input_sha256'].items():
    assert hashlib.sha256((root / relative_path).read_bytes()).hexdigest() == expected_hash

reviews = {record['pilot_id']: record for record in baseline['reviews']}
facts = {record['pilot_id']: record for record in evidence['records']}
for row in worksheet:
    prior = reviews[row['pilot_id']]
    observed = facts[row['pilot_id']]
    assert row['spotify_url'] == prior['spotify_url'] == observed['spotify']['url']
    assert row['youtube_url'] == prior['video_url'] == observed['youtube_preserved']['url']
    assert observed['prior_recording_review_status'] == prior['recording_review_status']
    assert abs(float(row['video_minus_snapshot_seconds']) -
               (float(row['youtube_duration_seconds_preserved']) - float(row['snapshot_duration_seconds']))) < 0.000001
    assert observed['spotify']['http_status'] == 200
    assert observed['audio_decision'] == row['final_recording_decision']
    if observed['audio_review_performed']:
        report = observed['audio_review']
        assert observed['audio_review_source'] == 'user_report_in_this_conversation'
        assert row['final_recording_decision'] in {'user_confirmed_match', 'user_reported_version_conflict', 'user_reported_video_audio_overlay'}
        assert row['reviewer'] and row['review_source']
        assert row['recorded_at_utc'] == report['recorded_at_utc']
        assert report['user_statement_verbatim'] in row['evidence_urls_or_notes']
        compared_urls = report.get('tested_urls', report.get('compared_urls'))
        assert compared_urls['spotify'] == row['spotify_url']
        assert compared_urls['youtube'] == row['youtube_url']
        if observed['audio_decision'] == 'user_reported_version_conflict':
            assert observed['main_sample_inclusion'] == 'blocked_version_conflict'
            durations = report['reported_durations_seconds']
            assert durations['youtube_minus_spotify'] == durations['youtube'] - durations['spotify']
        if report['reviewed_at_utc'] is None:
            assert not row['reviewed_at_utc']
    else:
        assert observed['audio_decision'] == 'pending'
        assert not row['reviewer'] and not row['reviewed_at_utc']

assert evidence['audio_reviews'] == sum(record['audio_review_performed'] for record in evidence['records'])
assert evidence['assistant_audio_reviews'] == 0

print('12 pares reconciliados; URLs exatas, durações e hashes de entrada conferidos.')
print('Estado original:', dict(Counter(record['prior_recording_review_status'] for record in evidence['records'])))
print('Páginas Spotify novas:', evidence['fresh_spotify_pages'])
print('Páginas YouTube novas:', evidence['fresh_youtube_pages'])
print('Comparações manuais relatadas pelo autor:', evidence['audio_reviews'])
print('Decisões atuais:', dict(Counter(record['audio_decision'] for record in evidence['records'])))
print('Audições pelo assistente:', evidence['assistant_audio_reviews'])
for pilot_id, fact in facts.items():
    if fact['audio_review_performed']:
        print(pilot_id + ' — decisão atual:', fact['audio_decision'])
        print(pilot_id + ' — relato:', fact['audio_review']['user_statement_verbatim'])
print('P02 — artistas Spotify:', facts['P02']['spotify']['facts']['music:musician_description'])
print('P03 — título YouTube preservado:', facts['P03']['youtube_preserved']['title'])
print('P06 — artistas Spotify:', facts['P06']['spotify']['facts']['music:musician_description'])
print('P06 — título YouTube preservado:', facts['P06']['youtube_preserved']['title'])

selection = json.loads((root / 'data/pilot/metadata-candidates.json').read_text())['candidates']
band_by_id = {record['pilot_id']: record['audience_band'] for record in selection}
summary = evidence['current_pairing_summary']
matched_ids = sorted(record['pilot_id'] for record in evidence['records'] if record['audio_decision'] == 'user_confirmed_match')
conflict_ids = sorted(record['pilot_id'] for record in evidence['records'] if record['overall_pairing_status'] == 'version_conflict')
overlay_ids = sorted(record['pilot_id'] for record in evidence['records'] if record['audio_decision'] == 'user_reported_video_audio_overlay')
assert summary['musically_confirmed_by_user']['pilot_ids'] == matched_ids
assert summary['musically_confirmed_by_user']['count'] == len(matched_ids)
assert summary['version_conflicts']['pilot_ids'] == conflict_ids
assert summary['version_conflicts']['count'] == len(conflict_ids)
assert summary['video_audio_overlay_pending']['pilot_ids'] == overlay_ids
assert len(matched_ids) + len(conflict_ids) + len(overlay_ids) == len(expected)
by_band = dict(sorted(Counter(band_by_id[pilot_id] for pilot_id in matched_ids).items()))
assert summary['confirmed_music_by_audience_band'] == by_band
checks = summary['recording_only_advance_checks']
assert checks['per_band_condition_met'] == all(by_band.get(band, 0) >= checks['minimum_per_band'] for band in set(band_by_id.values()))
assert summary['complete_case_count'] is None
print('Confirmações musicais por faixa:', by_band)
print('Critério de avanço atendido:', summary['advance_criterion_met'])
print('Motivo:', summary['reason'])


12 pares reconciliados; URLs exatas, durações e hashes de entrada conferidos.
Estado original: {'uncertain': 9, 'documentary_album_link': 2, 'version_conflict': 1}
Páginas Spotify novas: 12
Páginas YouTube novas: 0
Comparações manuais relatadas pelo autor: 11
Decisões atuais: {'user_confirmed_match': 9, 'user_reported_version_conflict': 1, 'pending': 1, 'user_reported_video_audio_overlay': 1}
Audições pelo assistente: 0
P01 — decisão atual: user_confirmed_match
P01 — relato: Correspondência exata
P02 — decisão atual: user_confirmed_match
P02 — relato: O tempo extra se explica no final da musica com um encerramento sem musica
De resto é idêntico do começo até esses segundos extras de encerramento do videoclipe
P03 — decisão atual: user_confirmed_match
P03 — relato: Confirmo que P03 é o mesmo no YouTube e no Spotify, testei os links e tem mesma duração e mesma música
P04 — decisão atual: user_confirmed_match
P04 — relato: a música a rigor é a mesma, o video no youtube tem uma intro de 10

A sequência proposta de escuta foi concluída: nove conteúdos musicais confirmados pelo autor, dois conflitos de versão (P05/P06) e P10 pendente por fala sobreposta. A distribuição 3/1/3/2 das confirmações por faixa não atende ao mínimo de dois na faixa de 1 a menos de 10 milhões. Gênero, datas e elegibilidade continuam pendentes; consultar [a consolidação e as fichas](../docs/conferencia-gravacoes.md).
